# Players → `nfl.players`

Reads raw ESPN blobs landed by Azure `teamsPlayersTrigger` (daily), parses them with `lib.transforms.parse_players`,
and upserts into `nfl.players` (newest `_snapshot_at` wins per key).
Schema, keys and column rules: `databricks/lib/schemas.py` and `databricks/TRANSFORMS.md`.

In [ ]:
import os
import sys

# Notebook CWD is databricks/notebooks/ in a Git folder; its parent holds the `lib` package.
sys.path.insert(0, os.path.abspath(".."))

from lib import delta_io, schemas, transforms

ACCOUNT = "footballmanagerli"
TABLE = schemas.TABLES["players"]
SRC = f"abfss://alsource@{ACCOUNT}.dfs.core.windows.net/{TABLE.source_prefix}/"
DEST = f"abfss://aldestination@{ACCOUNT}.dfs.core.windows.net/silver/{TABLE.name}"
CHECKPOINT = f"abfss://aldestination@{ACCOUNT}.dfs.core.windows.net/_checkpoints/silver_{TABLE.name}"

In [ ]:
delta_io.ensure_table(spark, TABLE, DEST)

raw = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("multiLine", "true")
    .schema(TABLE.raw_schema)
    .load(SRC)
)

parsed = transforms.parse_players(transforms.with_file_metadata(raw))

(
    parsed.writeStream.foreachBatch(delta_io.upsert_batch_fn(spark, TABLE))
    .option("checkpointLocation", CHECKPOINT)
    .trigger(availableNow=True)
    .start()
    .awaitTermination()
)

In [ ]:
keys = ", ".join(TABLE.keys)
counts = spark.sql(
    f"SELECT COUNT(*) AS row_count, COUNT(DISTINCT {keys}) AS key_count FROM {TABLE.full_name}"
).first()
print(counts)
assert counts.row_count == counts.key_count, "duplicate keys in table"

display(spark.table(TABLE.full_name).orderBy("_snapshot_at", ascending=False).limit(20))